In [51]:
# Core libraries
import pandas as pd
import numpy as np
from pathlib import Path
import logging

# ML libraries
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Simple logging setup
logging.basicConfig(
    level=logging.INFO,
    format='%(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

logger.info("Notebook initialized successfully")

INFO - Notebook initialized successfully


---

## 0. Data Scraping

Scape the data from https://store.steampowered.com/appreviews/ and store them in a csv file.

In [52]:
import requests
import pandas as pd
import time
from pathlib import Path
from typing import List, Dict, Optional
import logging

In [ ]:
def scrape_steam_reviews(app_id: int, max_reviews: int = 50000, language: str = 'english', reviews_per_page: int = 100) -> List[Dict]:
    all_reviews = []
    cursor = '*'
    start_time = time.time()
    
    while len(all_reviews) < max_reviews:
        url = f'https://store.steampowered.com/appreviews/{app_id}'
        params = {
            'json': 1,
            'language': language,
            'cursor': cursor,
            'num_per_page': reviews_per_page,
            'filter': 'recent'
        }
        
        try:
            response = requests.get(url, params=params, timeout=30)
            response.raise_for_status()
            data = response.json()
            
            if not data.get('reviews'):
                break
            
            all_reviews.extend(data['reviews'])
            cursor = data.get('cursor')
            
            if not cursor:
                break
            
            time.sleep(1)
            
        except requests.exceptions.RequestException as e:
            logging.error(f"Request failed: {e}")
            time.sleep(10)
            continue
    
    final_reviews = all_reviews[:max_reviews]
    elapsed_time = time.time() - start_time
    
    logging.info(f"Fetched {len(final_reviews)} reviews in {elapsed_time:.2f}s")
    
    return final_reviews

In [ ]:
def fetch_game_details(app_id: int) -> Dict:
    url = f'https://store.steampowered.com/api/appdetails'
    params = {'appids': app_id}
    
    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
        
        if str(app_id) in data and data[str(app_id)]['success']:
            game_data = data[str(app_id)]['data']
            return game_data
        else:
            return {}
            
    except requests.exceptions.RequestException as e:
        logging.error(f"Request failed: {e}")
        return {}


def process_reviews(reviews: List[Dict], game_data: Dict = None) -> pd.DataFrame:
    data = []
    
    for review in reviews:
        author = review.get('author', {})
        
        record = {
            'recommended': review.get('voted_up', False),
            'user_id': author.get('steamid', ''),
            'user_games_owned': author.get('num_games_owned', 0),
            'user_num_reviews': author.get('num_reviews', 0),
            'playtime_at_review_hours': author.get('playtime_at_review', 0) / 60,
            'playtime_total_hours': author.get('playtime_forever', 0) / 60,
            'playtime_recent_hours': author.get('playtime_last_two_weeks', 0) / 60,
            'received_free': review.get('received_for_free', False),
            'steam_purchase': review.get('steam_purchase', True),
            'written_early_access': review.get('written_during_early_access', False),
            'votes_helpful': review.get('votes_up', 0),
            'votes_funny': review.get('votes_funny', 0),
            'weighted_vote_score': review.get('weighted_vote_score', 0.0),
            'comment_count': review.get('comment_count', 0),
            'timestamp_created': review.get('timestamp_created', 0),
            'timestamp_updated': review.get('timestamp_updated', 0),
            'review_text': review.get('review', ''),
            'review_length': len(review.get('review', '')),
            'review_id': review.get('recommendationid', '')
        }
        
        if game_data:
            record.update({
                'game_id': game_data.get('steam_appid', ''),
                'game_name': game_data.get('name', ''),
                'game_price': game_data.get('price_overview', {}).get('final', 0) / 100 if game_data.get('price_overview') else 0,
                'game_is_free': game_data.get('is_free', False),
                'game_genre': ','.join([g['description'] for g in game_data.get('genres', [])]),
                'game_categories': ','.join([c['description'] for c in game_data.get('categories', [])]),
                'game_developer': ','.join(game_data.get('developers', [])),
                'game_publisher': ','.join(game_data.get('publishers', [])),
                'game_description': game_data.get('short_description', ''),
                'game_required_age': game_data.get('required_age', 0),
            })
        
        data.append(record)
    
    df = pd.DataFrame(data)
    
    df['playtime_ratio'] = df['playtime_at_review_hours'] / (df['playtime_total_hours'] + 1)
    df['review_engagement_score'] = df['votes_helpful'] + df['votes_funny'] * 0.5
    df['experienced_gamer'] = df['user_games_owned'] > 50
    df['active_reviewer'] = df['user_num_reviews'] > 5
    
    return df

In [ ]:
# # SINGLE GAME SCRAPING (COMMENTED OUT)
# APP_ID = 578080
# MAX_REVIEWS = 50000
# OUTPUT_FILE = 'steam_reviews.csv'
# game_data = fetch_game_details(APP_ID)
# reviews = scrape_steam_reviews(app_id=APP_ID, max_reviews=MAX_REVIEWS)

GAMES = [
    236430,
    1517290,
    1938090,
    1203220,
    2357570,
    976730,
    1551360,
    1294810,
    954850,
    1086940,
]

REVIEWS_PER_GAME = 2500
OUTPUT_FILE = 'steam_reviews_multi.csv'

all_game_dfs = []

for idx, app_id in enumerate(GAMES):
    logging.info(f"Game {idx+1}/{len(GAMES)} (ID: {app_id})")
    
    try:
        game_data = fetch_game_details(app_id)
        time.sleep(2)
        
        reviews = scrape_steam_reviews(app_id=app_id, max_reviews=REVIEWS_PER_GAME)
        
        if len(reviews) > 0:
            df_game = process_reviews(reviews, game_data=game_data)
            all_game_dfs.append(df_game)
        
        time.sleep(5)
        
    except Exception as e:
        logging.error(f"Failed game {app_id}: {e}")
        continue

if all_game_dfs:
    df = pd.concat(all_game_dfs, ignore_index=True)
    df.to_csv(OUTPUT_FILE, index=False)
    logging.info(f"Saved {len(df)} reviews from {len(all_game_dfs)} games to {OUTPUT_FILE}")
else:
    logging.error("No data collected")

In [56]:
# # SINGLE GAME PROCESSING (COMMENTED OUT)
# df = process_reviews(reviews, game_data=game_data)
# df.to_csv(OUTPUT_FILE, index=False)
# logging.info(f"Dataset saved to {OUTPUT_FILE}")

In [57]:
print(f"Dataset shape: {df.shape}")
print(f"Positive reviews: {df['recommended'].sum()} ({df['recommended'].mean()*100:.1f}%)")
print(f"Number of unique games: {df['game_id'].nunique()}")

print(f"\nGames in dataset:")
for game_id in df['game_id'].unique():
    game_df = df[df['game_id'] == game_id]
    game_name = game_df['game_name'].iloc[0]
    pos_pct = game_df['recommended'].mean() * 100
    print(f"  {game_name}: {len(game_df)} reviews ({pos_pct:.1f}% positive)")

print(f"\nFirst few rows:")
df.head()

Dataset shape: (21752, 33)
Positive reviews: 13985 (64.3%)
Number of unique games: 10

Games in dataset:
  DARK SOULS™ II: 2500 reviews (82.8% positive)
  Battlefield™ 2042: 2500 reviews (47.7% positive)
  Call of Duty®: 2500 reviews (43.6% positive)
  NARAKA: BLADEPOINT: 2500 reviews (80.7% positive)
  Overwatch® 2: 123 reviews (44.7% positive)
  Halo: The Master Chief Collection: 2500 reviews (83.1% positive)
  Forza Horizon 5: 2500 reviews (87.2% positive)
  Redfall: 1629 reviews (41.4% positive)
  Kerbal Space Program 2: 2500 reviews (9.6% positive)
  Baldur's Gate 3: 2500 reviews (95.7% positive)

First few rows:


,recommended,user_id,user_games_owned,user_num_reviews,playtime_at_review_hours,playtime_total_hours,playtime_recent_hours,received_free,steam_purchase,written_early_access,...,game_genre,game_categories,game_developer,game_publisher,game_description,game_required_age,playtime_ratio,review_engagement_score,experienced_gamer,active_reviewer
0,True,76561198053944201,145,9,0.233333,0.233333,0.000000,False,True,False,...,"Action,RPG","Single-player,Multi-player,Co-op,Steam Achieve...","FromSoftware, Inc.","BANDAI NAMCO Entertainment,FromSoftware, Inc.","Developed by FROM SOFTWARE, DARK SOULS™ II is ...",15,0.189189,0.0,True,True
1,True,76561199815144195,0,8,6.700000,9.833333,9.833333,False,True,False,...,"Action,RPG","Single-player,Multi-player,Co-op,Steam Achieve...","FromSoftware, Inc.","BANDAI NAMCO Entertainment,FromSoftware, Inc.","Developed by FROM SOFTWARE, DARK SOULS™ II is ...",15,0.618462,0.0,False,True
2,False,76561198004038352,1862,284,199.566667,199.566667,80.533333,False,True,False,...,"Action,RPG","Single-player,Multi-player,Co-op,Steam Achieve...","FromSoftware, Inc.","BANDAI NAMCO Entertainment,FromSoftware, Inc.","Developed by FROM SOFTWARE, DARK SOULS™ II is ...",15,0.995014,2.0,True,True
3,False,76561199117526837,222,4,87.583333,87.583333,6.983333,False,True,False,...,"Action,RPG","Single-player,Multi-player,Co-op,Steam Achieve...","FromSoftware, Inc.","BANDAI NAMCO Entertainment,FromSoftware, Inc.","Developed by FROM SOFTWARE, DARK SOULS™ II is ...",15,0.988711,0.0,True,False
4,False,76561199226288585,0,3,72.266667,87.850000,60.950000,False,True,False,...,"Action,RPG","Single-player,Multi-player,Co-op,Steam Achieve...","FromSoftware, Inc.","BANDAI NAMCO Entertainment,FromSoftware, Inc.","Developed by FROM SOFTWARE, DARK SOULS™ II is ...",15,0.813356,0.0,False,False


---

## 1. Data Loading & Exploration

Load the preprocessed Steam reviews dataset and perform exploratory analysis.

In [74]:
from pathlib import Path


In [ ]:
DATA_FILE = Path('steam_reviews_multi.csv')

if not DATA_FILE.exists():
    raise FileNotFoundError(f"Data file not found: {DATA_FILE}")

df = pd.read_csv(DATA_FILE)
logger.info(f"Loaded {len(df):,} reviews")

print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
print(f"Total records: {len(df):,}")
print(f"Positive reviews: {df['recommended'].sum():,} ({df['recommended'].mean()*100:.1f}%)")
print(f"Negative reviews: {(~df['recommended']).sum():,} ({(~df['recommended']).mean()*100:.1f}%)")

In [77]:
# Data quality check
print("DATA QUALITY REPORT")
print("="*60)
print("\nMissing values:")
print(df.isnull().sum())
print("\nData types:")
print(df.dtypes)
print("\nBasic statistics:")
df.describe()

DATA QUALITY REPORT

Missing values:
recommended                   0
user_id                       0
user_games_owned              0
user_num_reviews              0
playtime_at_review_hours      0
playtime_total_hours          0
playtime_recent_hours         0
received_free                 0
steam_purchase                0
written_early_access          0
votes_helpful                 0
votes_funny                   0
weighted_vote_score           0
comment_count                 0
timestamp_created             0
timestamp_updated             0
review_text                 101
review_length                 0
review_id                     0
game_id                       0
game_name                     0
game_price                    0
game_is_free                  0
game_genre                    0
game_categories               0
game_developer                0
game_publisher                0
game_description              0
game_required_age             0
playtime_ratio                0
rev

,user_id,user_games_owned,user_num_reviews,playtime_at_review_hours,playtime_total_hours,playtime_recent_hours,votes_helpful,votes_funny,weighted_vote_score,comment_count,timestamp_created,timestamp_updated,review_length,review_id,game_id,game_price,game_required_age,playtime_ratio,review_engagement_score
count,2.175200e+04,21752.000000,21752.000000,21752.000000,21752.000000,21752.000000,21752.000000,21752.000000,21752.000000,21752.000000,2.175200e+04,2.175200e+04,21752.000000,2.175200e+04,2.175200e+04,21752.000000,21752.000000,21752.000000,21752.000000
mean,7.656120e+16,108.104726,17.521193,145.875517,195.903593,4.899871,4.694649,0.697131,0.504184,0.119299,1.739118e+09,1.739704e+09,196.680811,1.875453e+08,1.198120e+06,79.764790,7.209544,0.728641,5.043214
std,6.003077e+08,482.340569,90.919525,361.135602,490.646221,15.714426,76.899000,13.045206,0.051435,2.802934,3.058882e+07,3.004833e+07,512.217452,2.719158e+07,4.663107e+05,89.326271,8.522069,0.278976,79.507638
min,7.656120e+16,0.000000,1.000000,0.083333,0.083333,0.000000,0.000000,0.000000,0.088055,0.000000,1.642037e+09,1.642117e+09,0.000000,1.078923e+08,2.364300e+05,0.000000,0.000000,0.000079,0.000000
25%,7.656120e+16,0.000000,2.000000,10.450000,18.583333,0.000000,0.000000,0.000000,0.500000,0.000000,1.725286e+09,1.728765e+09,14.000000,1.740675e+08,9.767300e+05,39.990000,0.000000,0.547439,0.000000
50%,7.656120e+16,0.000000,6.000000,37.016667,55.733333,0.000000,0.000000,0.000000,0.500000,0.000000,1.757005e+09,1.757032e+09,49.000000,2.035139e+08,1.203220e+06,39.990000,0.000000,0.840000,0.000000
75%,7.656120e+16,90.000000,16.000000,120.258333,158.779167,0.833333,1.000000,0.000000,0.500000,0.000000,1.759813e+09,1.759845e+09,162.000000,2.060935e+08,1.517290e+06,59.990000,18.000000,0.962736,1.000000
max,7.656120e+16,28230.000000,6260.000000,9834.766667,13544.850000,327.566667,8146.000000,1668.000000,0.978379,331.000000,1.762120e+09,1.762120e+09,8000.000000,2.082685e+08,2.357570e+06,249.000000,18.000000,0.999764,8213.500000


---

## 2. Feature Engineering

Prepare features from review text, playtime, and other variables for the binary classifier.

In [78]:
# Feature engineering imports
from textblob import TextBlob


In [ ]:
df['review_text'] = df['review_text'].fillna('')
df['word_count'] = df['review_text'].apply(lambda x: len(str(x).split()))

df['log_playtime_at_review'] = np.log1p(df['playtime_at_review_hours'])

df['game_price'] = df['game_price'].fillna(0)
df['game_genre'] = df['game_genre'].fillna('Unknown')
df['game_description'] = df['game_description'].fillna('')

df['game_description_length'] = df['game_description'].apply(lambda x: len(str(x).split()))

features = [
    'recommended',
    'log_playtime_at_review',
    'game_price',
    'game_genre',
    'game_description_length',
    'word_count'
]

df_subset = df[features].copy()

logger.info(f"Features: {df_subset.shape[1]-1} features, {df_subset.shape[0]} samples")

print(f"Dataset shape: {df_subset.shape}")
df_subset.head()

---

## 3. Model Training

Train the binary classifier to predict whether a user will recommend a game.

In [80]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
import time
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
def train_classifier(X_train, y_train, model=None):
    from sklearn.preprocessing import StandardScaler, OneHotEncoder
    from sklearn.compose import ColumnTransformer
    from sklearn.pipeline import Pipeline
    from sklearn.linear_model import LogisticRegression
    
    numeric = ['log_playtime_at_review', 'game_price', 'game_description_length', 'word_count']
    categorical = ['game_genre']
    
    preprocessor = ColumnTransformer([
        ('num', StandardScaler(), numeric),
        ('cat', OneHotEncoder(handle_unknown='ignore', max_categories=10), categorical)
    ])
    
    if model is None:
        model = LogisticRegression(random_state=42, max_iter=1000)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    
    pipeline.fit(X_train, y_train)
    
    return pipeline

In [ ]:
X = df_subset.drop('recommended', axis=1)
y = df_subset['recommended']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

logger.info(f"Train: {len(X_train)} samples, Test: {len(X_test)} samples")

logger.info("Training Logistic Regression...")
pipeline_lr = train_classifier(X_train, y_train, model=LogisticRegression(random_state=42, max_iter=1000))

logger.info("Training Random Forest...")
pipeline_rf = train_classifier(X_train, y_train, model=RandomForestClassifier(n_estimators=100, random_state=42))

logger.info("Training XGBoost...")
pipeline_xgb = train_classifier(X_train, y_train, model=XGBClassifier(random_state=42, eval_metric='logloss'))

logger.info("Training SVM...")
pipeline_svm = train_classifier(X_train, y_train, model=SVC(probability=True, random_state=42))

logger.info("Training complete")

In [ ]:
models_to_test = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss'),
    'SVM': SVC(probability=True, random_state=42)
}

results = []

for model_name, model in models_to_test.items():
    start_time = time.time()
    pipeline = train_classifier(X_train, y_train, model=model)
    training_time = time.time() - start_time
    
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    
    accuracy = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    
    results.append({
        'Model': model_name,
        'Accuracy': accuracy,
        'AUC-ROC': auc,
        'Training Time (s)': training_time
    })
    
    print(f"{model_name}: Accuracy={accuracy:.4f}, AUC={auc:.4f}, Time={training_time:.2f}s")

results_df = pd.DataFrame(results).sort_values('Accuracy', ascending=False)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Model Comparison', fontsize=16, fontweight='bold')

ax1 = axes[0, 0]
colors = sns.color_palette("husl", len(results_df))
bars = ax1.bar(range(len(results_df)), results_df['Accuracy'], color=colors, edgecolor='black')
ax1.set_ylabel('Accuracy')
ax1.set_title('Accuracy Comparison')
ax1.set_xticks(range(len(results_df)))
ax1.set_xticklabels(results_df['Model'], rotation=45, ha='right')
ax1.set_ylim([0.75, 0.85])
ax1.grid(axis='y', alpha=0.3)
for bar, val in zip(bars, results_df['Accuracy']):
    ax1.text(bar.get_x() + bar.get_width()/2, val + 0.002, f'{val:.4f}', ha='center', fontsize=9)

ax2 = axes[0, 1]
for i, (idx, row) in enumerate(results_df.iterrows()):
    ax2.scatter(row['Training Time (s)'], row['Accuracy'], s=300, c=[colors[i]], alpha=0.7, edgecolors='black')
    ax2.annotate(row['Model'], (row['Training Time (s)'], row['Accuracy']), xytext=(5, 5), textcoords='offset points', fontsize=9)
ax2.set_xlabel('Training Time (log scale)')
ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy vs Speed')
ax2.set_xscale('log')
ax2.grid(True, alpha=0.3)

ax3 = axes[1, 0]
sorted_time = results_df.sort_values('Training Time (s)', ascending=False)
bars = ax3.bar(range(len(sorted_time)), sorted_time['Training Time (s)'], color=colors, edgecolor='black')
ax3.set_ylabel('Time (log scale)')
ax3.set_title('Training Time')
ax3.set_yscale('log')
ax3.set_xticks(range(len(sorted_time)))
ax3.set_xticklabels(sorted_time['Model'], rotation=45, ha='right')
ax3.grid(axis='y', alpha=0.3)

ax4 = axes[1, 1]
norm_accuracy = (results_df['Accuracy'] - results_df['Accuracy'].min()) / (results_df['Accuracy'].max() - results_df['Accuracy'].min())
norm_auc = (results_df['AUC-ROC'] - results_df['AUC-ROC'].min()) / (results_df['AUC-ROC'].max() - results_df['AUC-ROC'].min())
norm_speed = 1 - ((results_df['Training Time (s)'] - results_df['Training Time (s)'].min()) / (results_df['Training Time (s)'].max() - results_df['Training Time (s)'].min()))
results_df['Overall'] = (norm_accuracy * 0.4 + norm_auc * 0.4 + norm_speed * 0.2)

x = np.arange(len(results_df))
width = 0.25
ax4.bar(x - width, norm_accuracy, width, label='Accuracy', color='steelblue', alpha=0.8, edgecolor='black')
ax4.bar(x, norm_auc, width, label='AUC', color='coral', alpha=0.8, edgecolor='black')
ax4.bar(x + width, norm_speed, width, label='Speed', color='lightgreen', alpha=0.8, edgecolor='black')
ax4.set_xlabel('Model')
ax4.set_ylabel('Normalized Score')
ax4.set_title('Overall Performance')
ax4.set_xticks(x)
ax4.set_xticklabels(results_df['Model'], rotation=45, ha='right')
ax4.legend(loc='upper left')
ax4.set_ylim([0, 1.1])
ax4.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nBest: {results_df.iloc[0]['Model']} ({results_df.iloc[0]['Accuracy']:.4f})")

logger.info(f"Best model: {results_df.iloc[0]['Model']}")

---

## 4. Prediction Function (with Counterfactuals)

Create a function that:
- Takes a user + game input
- Outputs binary prediction (yes/no they'll like it)
- If prediction is negative: generates counterfactual explanations

In [84]:
def predict_with_counterfactuals(model, user_features, threshold=0.5):
    prediction = model.predict(user_features)[0]
    prob = model.predict_proba(user_features)[0][1]

    counterfactuals = []
    if not prediction:
        feature_names = user_features.columns.tolist()
        counterfactuals = generate_counterfactuals(model, user_features, feature_names)

    return {
        'prediction': bool(prediction),
        'probability': float(prob),
        'counterfactuals': counterfactuals
    }

In [85]:
def generate_counterfactuals(model, user_features, feature_names):
    counterfactuals = []
    
    entity_modifiable = [
        'game_price',
        'game_description_length',
    ]
    
    for feature in entity_modifiable:
        if feature not in feature_names:
            continue
        
        modified = user_features.copy()
        original_val = modified[feature].values[0]
        
        if feature == 'game_price':
            test_values = [0, 4.99, 9.99, 14.99, 19.99]
            test_values = [v for v in test_values if abs(v - original_val) > 0.01]
        elif feature == 'game_description_length':
            test_values = [original_val + delta for delta in [5, 10, 20, 50]]
        else:
            continue
        
        for test_val in test_values:
            modified[feature] = test_val
            
            try:
                if model.predict(modified)[0]:
                    if feature == 'game_price':
                        if test_val == 0:
                            explanation = f"Make the game free-to-play (currently ${original_val:.2f})"
                        elif test_val < original_val:
                            explanation = f"Lower price to ${test_val:.2f} (currently ${original_val:.2f})"
                        else:
                            explanation = f"Adjust price to ${test_val:.2f} (currently ${original_val:.2f})"
                    elif feature == 'game_description_length':
                        delta = int(test_val - original_val)
                        explanation = f"Expand game description by {delta} words (currently {int(original_val)} words)"
                    
                    counterfactuals.append({
                        'feature': feature,
                        'original': float(original_val),
                        'suggested': float(test_val),
                        'change': explanation,
                        'type': 'entity',
                        'method': 'greedy'
                    })
                    break
            except:
                continue
    
    if 'game_genre' in feature_names:
        modified = user_features.copy()
        original_genre = modified['game_genre'].values[0]
        
        popular_additions = [
            'Action,RPG,Adventure',
            'Strategy,Simulation',
            'Multiplayer,Co-op',
            'Casual,Indie'
        ]
        
        for genre_test in popular_additions:
            if genre_test not in original_genre:
                modified['game_genre'] = original_genre + ',' + genre_test
                try:
                    if model.predict(modified)[0]:
                        explanation = f"Add genre tags: '{genre_test}' to appeal to broader audience"
                        counterfactuals.append({
                            'feature': 'game_genre',
                            'original': original_genre,
                            'suggested': modified['game_genre'].values[0],
                            'change': explanation,
                            'type': 'entity',
                            'method': 'greedy'
                        })
                        break
                except:
                    continue
    
    return counterfactuals

In [ ]:
def generate_counterfactuals_dice(model, user_features, feature_names, num_cfs=3):
    counterfactuals = []
    
    numeric_entity = ['game_price', 'game_description_length']
    available = [f for f in numeric_entity if f in feature_names]
    
    strategies = [
        {'game_price': 0},
        {'game_price': -5.0, 'game_description_length': 20},
        {'game_description_length': 50},
    ]
    
    for i, strategy in enumerate(strategies[:num_cfs]):
        modified = user_features.copy()
        changes = {}
        
        for feature, delta in strategy.items():
            if feature not in available:
                continue
            
            old_val = modified[feature].values[0]
            
            if feature == 'game_price':
                if delta == 0 or delta < 0 and abs(delta) > old_val:
                    new_val = max(0, delta) if delta >= 0 else max(0, old_val + delta)
                else:
                    new_val = old_val + delta
            else:
                new_val = old_val + delta
            
            modified[feature] = new_val
            changes[feature] = (float(old_val), float(new_val))
        
        try:
            if model.predict(modified)[0]:
                distance = np.sqrt(sum((new - old)**2 for old, new in changes.values()))
                
                descriptions = []
                for feature, (old_val, new_val) in changes.items():
                    if feature == 'game_price':
                        if new_val == 0:
                            descriptions.append("make free-to-play")
                        elif new_val < old_val:
                            descriptions.append(f"lower price (${old_val:.2f}→${new_val:.2f})")
                        else:
                            descriptions.append(f"adjust price (${old_val:.2f}→${new_val:.2f})")
                    elif feature == 'game_description_length':
                        delta = int(new_val - old_val)
                        descriptions.append(f"expand description by {delta} words")
                
                counterfactuals.append({
                    'explanation': " + ".join(descriptions),
                    'feature_changes': changes,
                    'distance': float(distance),
                    'type': 'entity',
                    'method': 'dice'
                })
        except:
            continue
    
    return counterfactuals

In [ ]:
def generate_anchors_explanation(model, user_features, feature_names, threshold=0.85):
    numeric = ['log_playtime_at_review', 'game_price', 'game_description_length', 'word_count']
    available = [f for f in numeric if f in feature_names]
    
    original_pred = model.predict(user_features)[0]
    
    feature_stability = []
    
    for feature in available:
        predictions = []
        
        for _ in range(50):
            test = user_features.copy()
            
            for other in available:
                if other != feature:
                    original = test[other].values[0]
                    noise = np.random.normal(0, 0.3)
                    test[other] = original + noise
            
            try:
                pred = model.predict(test)[0]
                predictions.append(pred)
            except:
                continue
        
        if predictions:
            stability = sum(p == original_pred for p in predictions) / len(predictions)
            value = user_features[feature].values[0]
            feature_stability.append((feature, stability, value))
    
    feature_stability.sort(key=lambda x: x[1], reverse=True)
    
    conditions = []
    for feature, stability, value in feature_stability[:3]:
        if stability >= threshold:
            if feature == 'log_playtime_at_review':
                conditions.append(f"playtime > {np.expm1(value):.0f} hours")
            elif feature == 'game_price':
                conditions.append(f"price < ${value:.0f}")
            elif feature == 'game_description_length':
                conditions.append(f"description > {int(value)} words")
            elif feature == 'word_count':
                conditions.append(f"review_length > {int(value)} words")
            else:
                conditions.append(f"{feature} > {value:.0f}")
    
    if conditions:
        rule = "IF " + " AND ".join(conditions)
        rule += f" THEN {'RECOMMENDED' if original_pred else 'NOT RECOMMENDED'}"
        
        test_preds = []
        for _ in range(100):
            test = user_features.copy()
            for f in available:
                if f not in [feat for feat, _, _ in feature_stability[:3]]:
                    test[f] = test[f].values[0] + np.random.normal(0, 0.5)
            try:
                test_preds.append(model.predict(test)[0])
            except:
                continue
        
        precision = sum(p == original_pred for p in test_preds) / len(test_preds) if test_preds else 0
        
        return {
            'anchor_rule': rule,
            'precision': float(precision),
            'num_conditions': len(conditions),
            'method': 'anchors'
        }
    else:
        return {
            'anchor_rule': 'No stable conditions found',
            'precision': 0.0,
            'num_conditions': 0,
            'method': 'anchors'
        }

In [ ]:
print("COUNTERFACTUAL ANALYSIS")
print("-" * 60)

test_user = pd.DataFrame({
    'log_playtime_at_review': [2.5],
    'game_price': [59.99],
    'game_genre': ['Action,Shooter'],
    'game_description_length': [15],
    'word_count': [10]
})

pred = pipeline_lr.predict(test_user)[0]
prob = pipeline_lr.predict_proba(test_user)[0][1]

print(f"\nUser: {np.expm1(test_user['log_playtime_at_review'].values[0]):.0f}h playtime, {test_user['word_count'].values[0]} word review")
print(f"Game: ${test_user['game_price'].values[0]:.2f}, {test_user['game_genre'].values[0]}, {test_user['game_description_length'].values[0]} word description")
print(f"Prediction: {'RECOMMENDED' if pred else 'NOT RECOMMENDED'} ({prob:.1%})")

print(f"\nGreedy Counterfactuals:")
greedy = generate_counterfactuals(pipeline_lr, test_user, test_user.columns.tolist())
for i, cf in enumerate(greedy, 1):
    print(f"  {i}. {cf['change']}")

print(f"\nDICE Counterfactuals:")
dice = generate_counterfactuals_dice(pipeline_lr, test_user, test_user.columns.tolist())
for i, cf in enumerate(dice, 1):
    print(f"  {i}. {cf['explanation']}")

print(f"\nAnchor Rules:")
positive_user = pd.DataFrame({
    'log_playtime_at_review': [5.0],
    'game_price': [19.99],
    'game_genre': ['Action,RPG'],
    'game_description_length': [40],
    'word_count': [50]
})
anchor = generate_anchors_explanation(pipeline_lr, positive_user, positive_user.columns.tolist())
print(f"  {anchor['anchor_rule']} (precision: {anchor['precision']:.1%})")

## TODO: Add observations here